# 🧩 Notebook 3: Blackjack — Real-world extensions


## 🛠️ Setup

```bash
cd 07-object-oriented-design/blackjack
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎯 What you'll learn

The core game from Notebook 2 works, but it's still a toy. Real casinos (and real software!) have to deal with: *betting*, *different playing styles*, and *splits*. Each extension teaches a classic OO lesson:

1. **Betting** → composition (a `Player` *has a* wallet).
2. **Strategies** → Strategy Pattern (composition over inheritance).
3. **Splits** → revisiting the domain model when requirements grow.


## 1️⃣ Re-usable core (copied from Notebook 2)

So this notebook is self-contained and runnable on its own.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from abc import ABC, abstractmethod
import random

class Suit(Enum):
    HEARTS = "♥"; DIAMONDS = "♦"; CLUBS = "♣"; SPADES = "♠"

RANKS = ["A","2","3","4","5","6","7","8","9","10","J","Q","K"]

@dataclass(frozen=True)
class Card:
    rank: str
    suit: Suit
    def value(self) -> int:
        if self.rank == "A":           return 11
        if self.rank in ("J","Q","K"): return 10
        return int(self.rank)
    def __repr__(self): return f"{self.rank}{self.suit.value}"

@dataclass
class Deck:
    cards: list[Card] = field(default_factory=list)
    def __post_init__(self):
        if not self.cards:
            self.cards = [Card(r, s) for s in Suit for r in RANKS]
    def shuffle(self):      random.shuffle(self.cards)
    def draw(self) -> Card: return self.cards.pop()

@dataclass
class Hand:
    cards: list[Card] = field(default_factory=list)
    def add(self, c): self.cards.append(c)
    def value(self) -> int:
        total = sum(c.value() for c in self.cards)
        aces  = sum(1 for c in self.cards if c.rank == "A")
        while total > 21 and aces:
            total -= 10; aces -= 1
        return total
    def is_bust(self):       return self.value() > 21
    def is_blackjack(self):  return len(self.cards) == 2 and self.value() == 21
    def __repr__(self):      return f"{self.cards} = {self.value()}"

print("Core classes loaded.")


## 2️⃣ The Strategy Pattern — swap playing styles without subclassing

In Notebook 2, `Dealer` *inherited* from `Player` just to change one method. That works for two kinds of seat. But imagine 5 playing styles (Conservative, Aggressive, BasicStrategyChart, …). We'd have a deep class tree just to vary one decision.

The **Strategy Pattern** fixes this: pull the decision out into its own object, and let `Player` *hold* one. This is the textbook example of *composition over inheritance*.


In [ ]:
class HitStrategy(ABC):
    """Any object that can answer: given this hand, should I hit?"""
    @abstractmethod
    def wants_hit(self, hand: Hand) -> bool: ...

class HitUntil(HitStrategy):
    """Hit while the hand value is below `threshold`."""
    def __init__(self, threshold: int): self.threshold = threshold
    def wants_hit(self, hand): return hand.value() < self.threshold

class NeverHit(HitStrategy):
    def wants_hit(self, hand): return False

class AggressiveThenFold(HitStrategy):
    """Hit once if under 19, then stand. Illustrative — not real basic strategy."""
    def __init__(self): self._hit_once = False
    def wants_hit(self, hand):
        if self._hit_once or hand.value() >= 19: return False
        self._hit_once = True
        return True

class Player:
    def __init__(self, name: str, strategy: HitStrategy):
        self.name = name
        self.hand = Hand()
        self.strategy = strategy
    def wants_hit(self) -> bool:
        return self.strategy.wants_hit(self.hand)

# Dealer is now just a Player with a fixed strategy — no subclass needed!
def make_dealer() -> Player:
    return Player("Dealer", HitUntil(17))

print("Strategies ready.")


## 3️⃣ Betting — `Chips` via composition

A `Player` *has a* wallet. We don't make `Player` a subclass of `Wallet` — that would be silly. This is another composition win.


In [ ]:
class Chips:
    def __init__(self, balance: int = 100):
        self.balance = balance
        self.current_bet = 0
    def place_bet(self, amount: int):
        if amount > self.balance:
            raise ValueError(f"Not enough chips: {self.balance} < {amount}")
        self.balance -= amount
        self.current_bet = amount
    def win(self, multiplier: float = 2.0):
        # Standard win returns 2x the bet (original + equal amount)
        self.balance += int(self.current_bet * multiplier)
        self.current_bet = 0
    def push(self):
        # Tie — get the original bet back
        self.balance += self.current_bet
        self.current_bet = 0
    def lose(self):
        self.current_bet = 0
    def __repr__(self): return f"Chips(balance={self.balance}, bet={self.current_bet})"

class BettingPlayer(Player):
    def __init__(self, name, strategy, chips: Chips):
        super().__init__(name, strategy)
        self.chips = chips

# Quick demo of chip math
c = Chips(100)
c.place_bet(20);         print(c)
c.win(multiplier=2.5);   print(c, "(blackjack pays 3:2)")


## 4️⃣ A table that puts it all together


In [ ]:
class Table:
    """Runs rounds with betting, pluggable strategies, and a dealer."""
    def __init__(self, players: list[BettingPlayer]):
        self.players = players
        self.dealer  = make_dealer()

    def play_round(self, bet: int = 10):
        # Fresh deck every round keeps the example simple
        deck = Deck(); deck.shuffle()
        for p in self.players:
            p.hand = Hand()
            p.chips.place_bet(bet)
        self.dealer.hand = Hand()

        # Initial deal
        for _ in range(2):
            for p in self.players + [self.dealer]:
                p.hand.add(deck.draw())

        # Player turns
        for p in self.players:
            while p.wants_hit() and not p.hand.is_bust():
                p.hand.add(deck.draw())
        # Dealer turn
        while self.dealer.wants_hit() and not self.dealer.hand.is_bust():
            self.dealer.hand.add(deck.draw())

        # Settle
        d = self.dealer.hand.value()
        print(f"Dealer: {self.dealer.hand}")
        for p in self.players:
            pv = p.hand.value()
            if   p.hand.is_bust():              p.chips.lose();  outcome = "BUST"
            elif p.hand.is_blackjack():         p.chips.win(2.5); outcome = "BLACKJACK!"
            elif self.dealer.hand.is_bust():    p.chips.win();   outcome = "WIN"
            elif pv >  d:                       p.chips.win();   outcome = "WIN"
            elif pv == d:                       p.chips.push();  outcome = "PUSH"
            else:                               p.chips.lose();  outcome = "LOSE"
            print(f"  {p.name} ({pv}) {outcome}  {p.chips}")

random.seed(42)
table = Table([
    BettingPlayer("Alice",  HitUntil(17),          Chips(100)),
    BettingPlayer("Bob",    NeverHit(),            Chips(100)),
    BettingPlayer("Carol",  AggressiveThenFold(),  Chips(100)),
])
for round_no in range(1, 4):
    print(f"\n===== Round {round_no} =====")
    table.play_round(bet=10)


## 5️⃣ When requirements grow: splits (discussion)

If the first two cards are the same rank (say two 8s), real Blackjack lets you **split** the hand into two hands, each with its own bet. Notice what breaks in our current model:

- `Player` has *one* `hand` — but now a player can have *many*.
- `Chips.current_bet` is a single number — but each split hand needs its own bet.

**How would you redesign?** A common refactor:

```
Player ──has──▶ list[HandAndBet]
                  │
                  └── Hand + the wager on that particular hand
```

This is the **Open/Closed Principle** (`O` in SOLID) in action: the *design* stays open to extension (splits, doubles, insurance) without rewriting the core classes.

> 🧪 **Challenge:** implement `split()` on `BettingPlayer` and extend `Table` to loop over each split hand. Try it — then compare your approach to the discussion above.


## 🧭 Recap — the OO toolkit we used

| Tool | Where we used it | Why it helps |
|---|---|---|
| **Single Responsibility** | Ace rule lives in `Hand` | One reason to change |
| **Polymorphism** | `wants_hit()` on every seat | `Game` treats everyone the same |
| **Strategy Pattern** | `HitStrategy` objects | Add playing styles without new classes |
| **Composition over Inheritance** | `Player` *has a* `Strategy` and `Chips` | Flexible, testable, less code |
| **Encapsulation** | `Chips` owns balance & bet math | Callers can't corrupt state |
| **Open/Closed** | Splits/doubles as future extensions | Grow the game without rewriting it |
